# Punto 7 - Análisis descriptivo de Saber 11 con PySpark

Responde las 7 preguntas de negocio usando **solo la API de DataFrames de PySpark**
(sin SQL), sobre las tablas de la zona *trusted*.

- En **EMR** (EMR Studio o JupyterHub): cambie `TRUSTED` por `s3://SU_BUCKET/trusted`.
- En **local**: deje la ruta de la muestra de prueba.

Los gráficos se guardan en la carpeta `graficos/` para el documento.

Notas de método:
- Los promedios territoriales se ponderan por número de evaluados.
- "Último periodo" y la evolución anual usan solo **calendario A**: los periodos de calendario B
  evalúan ~340 colegios, casi todos privados bilingües, y distorsionan rankings y tendencias.
- `es_sintetico` marca los periodos cuyo archivo del ICFES llegó corrupto y se reemplazó con datos
  sintéticos (generate_synthetic_icfes.py).

> **Atención:** las salidas guardadas en este notebook y los gráficos actuales de `graficos/`
> salen de una **prueba local** (archivos sintéticos del repositorio). Se reemplazan al ejecutar
> el notebook en EMR sobre el bucket real.

In [1]:
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

TRUSTED = os.environ.get("TRUSTED", "../pruebas-locales/prueba/trusted")   # en EMR: "s3://SU_BUCKET/trusted"
GRAFICOS = os.environ.get("GRAFICOS", "graficos")
os.makedirs(GRAFICOS, exist_ok=True)

spark = (SparkSession.builder.appName("saber11-pyspark")
         .config("spark.sql.sources.partitionColumnTypeInference.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

cp = spark.read.parquet(f"{TRUSTED}/colegio_periodo").cache()
try:
    est = spark.read.parquet(f"{TRUSTED}/estudiante").cache()   # fuente opcional (P5)
except Exception:
    est = None

ultimo = cp.filter(F.col("semestre") != 1).agg(F.max("periodo")).first()[0]   # último de calendario A
cp_ult = cp.filter(F.col("periodo") == ultimo)
print(f"Colegio-periodo: {cp.count()} filas | Estudiantes: {est.count() if est is not None else 'no cargados'}"
      f" | Último periodo (calendario A): {ultimo}")
cp.groupBy("periodo", "es_sintetico").agg(F.count("*").alias("filas"), F.sum("evaluados").alias("evaluados")) \
  .orderBy("periodo").show(40)


def prom_ponderado(col, peso="evaluados"):
    """Promedio ponderado por número de evaluados."""
    return F.round(F.sum(F.col(col) * F.col(peso)) / F.sum(peso), 2)

Colegio-periodo: 91282 filas | Estudiantes: no cargados | Último periodo (calendario A): 20234


+-------+------------+-----+---------+
|periodo|es_sintetico|filas|evaluados|
+-------+------------+-----+---------+
|  20122|        true|12966|   506931|
|  20162|        true|12925|   507914|
|  20172|        true|12963|   508230|
|  20181|        true|  342|    12193|
|  20191|        true|  335|    12171|
|  20192|        true|12930|   506358|
|  20204|        true|12945|   507347|
|  20214|        true|12938|   508715|
|  20234|       false|12938|   508715|
+-------+------------+-----+---------+



## P1. Departamentos y municipios con mayor y menor puntaje global

In [2]:
p1_depto = (cp_ult.groupBy("departamento", "region")
                  .agg(F.sum("evaluados").alias("evaluados"), prom_ponderado("prom_global").alias("prom_global"))
                  .orderBy(F.desc("prom_global")))
p1_depto.show(40, truncate=False)

p1_mpio = (cp_ult.groupBy("departamento", "municipio")
                 .agg(F.sum("evaluados").alias("evaluados"), prom_ponderado("prom_global").alias("prom_global"))
                 .filter(F.col("evaluados") >= 100))
print("Mejores municipios:")
p1_mpio.orderBy(F.desc("prom_global")).show(10, truncate=False)
print("Peores municipios:")
p1_mpio.orderBy("prom_global").show(10, truncate=False)

pdf = p1_depto.toPandas().sort_values("prom_global")
fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(pdf))))
ax.barh(pdf["departamento"], pdf["prom_global"], color="#2b6c8f")
ax.set_xlabel("Puntaje global promedio (0-500)")
ax.set_title(f"P1. Puntaje global por departamento, periodo {ultimo}")
fig.tight_layout(); fig.savefig(f"{GRAFICOS}/p1_departamentos.png", dpi=150); plt.close(fig)

+------------------+---------+---------+-----------+
|departamento      |region   |evaluados|prom_global|
+------------------+---------+---------+-----------+
|BOGOTA D.C.       |ANDINA   |72520    |266.44     |
|BOYACA            |ANDINA   |15285    |265.69     |
|SANTANDER         |ANDINA   |24882    |264.86     |
|QUINDIO           |ANDINA   |5059     |263.86     |
|CUNDINAMARCA      |ANDINA   |36267    |260.62     |
|NORTE DE SANTANDER|ANDINA   |16416    |259.65     |
|CASANARE          |ORINOQUIA|5470     |255.79     |
|HUILA             |ANDINA   |12378    |254.94     |
|META              |ORINOQUIA|11493    |252.34     |
|CALDAS            |ANDINA   |9172     |251.3      |
|RISARALDA         |ANDINA   |9370     |250.36     |
|NARIÑO            |PACIFICA |14432    |247.66     |
|ATLANTICO         |CARIBE   |30579    |246.58     |
|VALLE DEL CAUCA   |PACIFICA |34751    |246.53     |
|TOLIMA            |ANDINA   |15382    |244.74     |
|ANTIOQUIA         |ANDINA   |68880    |244.18

+------------------+---------------------+---------+-----------+
|departamento      |municipio            |evaluados|prom_global|
+------------------+---------------------+---------+-----------+
|CUNDINAMARCA      |COTA                 |929      |304.57     |
|BOYACA            |PAIPA                |427      |297.4      |
|BOYACA            |VILLA DE LEYVA       |145      |293.24     |
|BOYACA            |NOBSA                |167      |286.04     |
|NORTE DE SANTANDER|PAMPLONA             |424      |284.99     |
|ANTIOQUIA         |SABANETA             |752      |282.42     |
|CUNDINAMARCA      |TENJO                |470      |281.58     |
|BOYACA            |SANTA ROSA DE VITERBO|118      |281.28     |
|SANTANDER         |MALAGA               |317      |279.9      |
|BOYACA            |DUITAMA              |1861     |279.82     |
+------------------+---------------------+---------+-----------+
only showing top 10 rows

Peores municipios:


+------------+-----------------------+---------+-----------+
|departamento|municipio              |evaluados|prom_global|
+------------+-----------------------+---------+-----------+
|CHOCO       |EL LITORAL DEL SAN JUAN|152      |169.31     |
|CAUCA       |TIMBIQUI               |187      |181.68     |
|CHOCO       |BAJO BAUDO             |126      |183.88     |
|CAUCA       |LOPEZ DE MICAY         |104      |184.54     |
|MAGDALENA   |PUEBLOVIEJO            |232      |186.75     |
|LA GUAJIRA  |URIBIA                 |546      |188.09     |
|NARIÑO      |EL CHARCO              |159      |188.22     |
|NARIÑO      |BARBACOAS              |377      |191.14     |
|CHOCO       |BAHIA SOLANO           |145      |191.18     |
|LA GUAJIRA  |MANAURE                |478      |191.2      |
+------------+-----------------------+---------+-----------+
only showing top 10 rows



## P2. Oficiales vs no oficiales según la zona
La zona (urbana/rural) viene del directorio de sedes de RDS, unido por el código DANE de la sede.

In [3]:
p2 = (cp_ult.filter(F.col("zona").isNotNull() & F.col("naturaleza").isNotNull())
            .groupBy("zona")
            .pivot("naturaleza", ["OFICIAL", "NO OFICIAL"])
            .agg(prom_ponderado("prom_global"))
            .withColumn("brecha", F.round(F.col("NO OFICIAL") - F.col("OFICIAL"), 2)))
print(f"Periodo {ultimo}. Filas con zona asignada: "
      f"{cp_ult.filter(F.col('zona').isNotNull()).count()} de {cp_ult.count()}")
p2.show(truncate=False)

pdf = p2.toPandas().set_index("zona")[["OFICIAL", "NO OFICIAL"]]
ax = pdf.plot(kind="bar", figsize=(7.5, 4), color=["#2b6c8f", "#e0a03a"], rot=0)
ax.set_ylabel("Puntaje global promedio")
ax.set_xlabel("")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"P2. Puntaje por naturaleza del colegio y zona, periodo {ultimo}")
plt.tight_layout(); plt.savefig(f"{GRAFICOS}/p2_naturaleza_zona.png", dpi=150); plt.close()

Periodo 20234. Filas con zona asignada: 12720 de 12938


+------+-------+----------+------+
|zona  |OFICIAL|NO OFICIAL|brecha|
+------+-------+----------+------+
|RURAL |237.74 |264.31    |26.57 |
|URBANA|248.12 |271.86    |23.74 |
+------+-------+----------+------+



## P3. Área con peor desempeño por región

In [4]:
areas = {"prom_lectura": "LECTURA CRITICA", "prom_matematicas": "MATEMATICAS",
         "prom_sociales": "SOCIALES Y CIUDADANAS", "prom_naturales": "CIENCIAS NATURALES",
         "prom_ingles": "INGLES"}
p3 = cp_ult.groupBy("region").agg(*[prom_ponderado(c).alias(c) for c in areas])
# Se pasa a formato largo (región, área, puntaje) y se toma el mínimo por región.
largo = p3.select("region", F.explode(F.array(*[
    F.struct(F.lit(n).alias("area"), F.col(c).alias("puntaje")) for c, n in areas.items()])).alias("x")
).select("region", "x.area", "x.puntaje")
w = Window.partitionBy("region").orderBy("puntaje")
p3_min = largo.withColumn("pos", F.row_number().over(w)).filter("pos = 1").drop("pos")
p3.show(truncate=False)
p3_min.orderBy("region").show(truncate=False)

pdf = p3.toPandas().set_index("region").rename(columns=areas)
ax = pdf.plot(kind="bar", figsize=(10, 4.5), rot=0)
ax.set_ylabel("Puntaje promedio (0-100)")
ax.set_xlabel("")
ax.set_title(f"P3. Puntaje por área y región, periodo {ultimo}")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.savefig(f"{GRAFICOS}/p3_areas_region.png", dpi=150); plt.close()

+---------+------------+----------------+-------------+--------------+-----------+
|region   |prom_lectura|prom_matematicas|prom_sociales|prom_naturales|prom_ingles|
+---------+------------+----------------+-------------+--------------+-----------+
|AMAZONIA |49.97       |48.54           |44.7         |46.86         |46.52      |
|ORINOQUIA|52.51       |51.74           |47.01        |49.61         |49.77      |
|CARIBE   |50.35       |48.02           |44.51        |47.17         |47.82      |
|PACIFICA |50.92       |48.94           |45.47        |47.78         |48.29      |
|ANDINA   |53.85       |52.41           |48.73        |50.35         |52.27      |
|INSULAR  |49.44       |44.7            |43.59        |44.5          |55.42      |
+---------+------------+----------------+-------------+--------------+-----------+



+---------+---------------------+-------+
|region   |area                 |puntaje|
+---------+---------------------+-------+
|AMAZONIA |SOCIALES Y CIUDADANAS|44.7   |
|ANDINA   |SOCIALES Y CIUDADANAS|48.73  |
|CARIBE   |SOCIALES Y CIUDADANAS|44.51  |
|INSULAR  |SOCIALES Y CIUDADANAS|43.59  |
|ORINOQUIA|SOCIALES Y CIUDADANAS|47.01  |
|PACIFICA |SOCIALES Y CIUDADANAS|45.47  |
+---------+---------------------+-------+



## P4. Evolución del puntaje por departamento

In [5]:
p4 = (cp.filter(F.col("semestre") != 1)          # solo calendario A
        .groupBy("departamento", "anio")
        .agg(prom_ponderado("prom_global").alias("prom_global"),
             F.max("es_sintetico").alias("incluye_sintetico"))
        .withColumn("variacion", F.round(
            F.col("prom_global") - F.lag("prom_global").over(Window.partitionBy("departamento").orderBy("anio")), 2)))
p4.orderBy("departamento", "anio").show(60, truncate=False)

# Para que el gráfico se lea: promedio nacional + los 3 mejores y 3 peores departamentos del último año.
nacional = (cp.filter(F.col("semestre") != 1).groupBy("anio")
              .agg(prom_ponderado("prom_global").alias("prom_global"), F.max("es_sintetico").alias("sint"))
              .orderBy("anio").toPandas().set_index("anio"))
tabla = p4.toPandas().pivot(index="anio", columns="departamento", values="prom_global").sort_index()
ultimo_anio = tabla.index.max()
orden = tabla.loc[ultimo_anio].dropna().sort_values()
elegidos = list(orden.index[-3:][::-1]) + list(orden.index[:3])
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(nacional.index, nacional["prom_global"], color="black", linewidth=3, marker="o", label="NACIONAL")
for d in elegidos:
    ax.plot(tabla.index, tabla[d], marker="o", linewidth=1.5, label=d)
etiquetas = [f"{a}*" if nacional.loc[a, "sint"] else str(a) for a in nacional.index]
ax.set_xticks(nacional.index); ax.set_xticklabels(etiquetas)
ax.set_ylabel("Puntaje global promedio")
ax.set_xlabel("Año   (* = periodo con datos sintéticos)")
ax.set_title("P4. Evolución del puntaje global: nacional, 3 mejores y 3 peores departamentos")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout(); fig.savefig(f"{GRAFICOS}/p4_evolucion.png", dpi=150); plt.close(fig)

+------------+----+-----------+-----------------+---------+
|departamento|anio|prom_global|incluye_sintetico|variacion|
+------------+----+-----------+-----------------+---------+
|AMAZONAS    |2012|230.66     |true             |NULL     |
|AMAZONAS    |2016|226.82     |true             |-3.84    |
|AMAZONAS    |2017|228.7      |true             |1.88     |
|AMAZONAS    |2019|223.4      |true             |-5.3     |
|AMAZONAS    |2020|214.76     |true             |-8.64    |
|AMAZONAS    |2021|213.5      |true             |-1.26    |
|AMAZONAS    |2023|213.5      |false            |0.0      |
|ANTIOQUIA   |2012|257.58     |true             |NULL     |
|ANTIOQUIA   |2016|255.23     |true             |-2.35    |
|ANTIOQUIA   |2017|254.88     |true             |-0.35    |
|ANTIOQUIA   |2019|252.73     |true             |-2.15    |
|ANTIOQUIA   |2020|246.61     |true             |-6.12    |
|ANTIOQUIA   |2021|244.18     |true             |-2.43    |
|ANTIOQUIA   |2023|244.18     |false    

## P5. Estrato, educación de los padres y puntaje

In [6]:
if est is None:
    print("P5 no se puede calcular: requiere datos por estudiante (estrato y educación de los padres),\n"
          "que no vienen en los archivos agregados por colegio. Ver README del punto 7.")
else:
    p5_estrato = (est.filter(F.col("estrato").isNotNull()).groupBy("estrato")
                     .agg(F.count("*").alias("estudiantes"), F.round(F.avg("punt_global"), 2).alias("prom_global"))
                     .orderBy("estrato"))
    p5_padres = (est.filter(F.col("nivel_educ_padres").isNotNull()).groupBy("nivel_educ_padres_txt")
                    .agg(F.count("*").alias("estudiantes"), F.round(F.avg("punt_global"), 2).alias("prom_global"))
                    .orderBy("nivel_educ_padres_txt"))
    p5_estrato.show(truncate=False)
    p5_padres.show(truncate=False)

    # stat.corr no descarta nulos por sí solo: se filtran antes para que coincida con CORR() de SQL.
    corr_estrato = est.filter(F.col("estrato").isNotNull()).stat.corr("estrato", "punt_global")
    corr_padres = est.filter(F.col("nivel_educ_padres").isNotNull()).stat.corr("nivel_educ_padres", "punt_global")
    print(f"Correlación estrato-puntaje: {corr_estrato:.3f}")
    print(f"Correlación educación padres-puntaje: {corr_padres:.3f}")

    matriz = (est.filter(F.col("estrato").isNotNull() & F.col("nivel_educ_padres").isNotNull())
                 .groupBy("estrato").pivot("nivel_educ_padres").agg(F.round(F.avg("punt_global"), 1))
                 .orderBy("estrato").toPandas().set_index("estrato"))
    fig, ax = plt.subplots(figsize=(7, 4.5))
    im = ax.imshow(matriz.values, cmap="YlGnBu", aspect="auto")
    ax.set_xticks(range(len(matriz.columns))); ax.set_xticklabels(matriz.columns)
    ax.set_yticks(range(len(matriz.index))); ax.set_yticklabels(matriz.index)
    ax.set_xlabel("Máximo nivel educativo de los padres (0 ninguno - 5 posgrado)")
    ax.set_ylabel("Estrato")
    ax.set_title("P5. Puntaje global promedio por estrato y educación de los padres")
    fig.colorbar(im, ax=ax, label="Puntaje global")
    fig.tight_layout(); fig.savefig(f"{GRAFICOS}/p5_estrato_padres.png", dpi=150); plt.close(fig)

P5 no se puede calcular: requiere datos por estudiante (estrato y educación de los padres),
que no vienen en los archivos agregados por colegio. Ver README del punto 7.


## P6. Top 10 colegios por departamento y concentración por municipio

In [7]:
colegio = (cp_ult.groupBy("departamento", "municipio", "cod_colegio", "naturaleza")
                 .agg(F.first("nombre_colegio", ignorenulls=True).alias("nombre_colegio"),
                      F.sum("evaluados").alias("evaluados"),
                      prom_ponderado("prom_global").alias("prom_global"))
                 .filter(F.col("evaluados") >= 10))
w = Window.partitionBy("departamento").orderBy(F.desc("prom_global"), F.desc("evaluados"))
top10 = colegio.withColumn("puesto", F.row_number().over(w)).filter(F.col("puesto") <= 10)
top10.orderBy("departamento", "puesto").show(40, truncate=False)

concentracion = (top10.groupBy("departamento", "municipio").count()
                      .withColumnRenamed("count", "colegios_en_top10")
                      .orderBy("departamento", F.desc("colegios_en_top10")))
concentracion.show(40, truncate=False)

print("Naturaleza de los colegios del top 10:")
top10.groupBy("naturaleza").count().show()

+------------+---------------+-----------+----------+----------------------------------------------------------------------------------------------+---------+-----------+------+
|departamento|municipio      |cod_colegio|naturaleza|nombre_colegio                                                                                |evaluados|prom_global|puesto|
+------------+---------------+-----------+----------+----------------------------------------------------------------------------------------------+---------+-----------+------+
|AMAZONAS    |LETICIA        |087429     |NO OFICIAL|COLEGIO CRISTO REY MINUTO DE DIOS                                                             |23       |280.77     |1     |
|AMAZONAS    |LETICIA        |780379     |NO OFICIAL|LICEO LOS ANGELES - SEDE PRINCIPAL                                                            |16       |273.85     |2     |
|AMAZONAS    |LETICIA        |717249     |OFICIAL   |I.E. INEM JOSE EUSTASIO RIVERA - SEDE PRINCIPAL          

+------------+----------------------+-----------------+
|departamento|municipio             |colegios_en_top10|
+------------+----------------------+-----------------+
|AMAZONAS    |LETICIA               |8                |
|AMAZONAS    |EL ENCANTO            |1                |
|AMAZONAS    |PUERTO NARIÑO         |1                |
|ANTIOQUIA   |MEDELLIN              |5                |
|ANTIOQUIA   |RIONEGRO              |1                |
|ANTIOQUIA   |ITAGUI                |1                |
|ANTIOQUIA   |MARINILLA             |1                |
|ANTIOQUIA   |GIRARDOTA             |1                |
|ANTIOQUIA   |SABANETA              |1                |
|ARAUCA      |ARAUCA                |5                |
|ARAUCA      |SARAVENA              |2                |
|ARAUCA      |TAME                  |2                |
|ARAUCA      |ARAUQUITA             |1                |
|ATLANTICO   |BARRANQUILLA          |9                |
|ATLANTICO   |PUERTO COLOMBIA       |1          

+----------+-----+
|naturaleza|count|
+----------+-----+
|   OFICIAL|  107|
|NO OFICIAL|  217|
+----------+-----+



## P7. Brecha entre Bogotá, Medellín, Cali y el resto del país

In [8]:
grupo = (F.when(F.col("cod_mpio") == "11001", "BOGOTA D.C.")
          .when(F.col("cod_mpio") == "05001", "MEDELLIN")
          .when(F.col("cod_mpio") == "76001", "CALI")
          .otherwise("RESTO DEL PAIS"))
p7 = (cp.withColumn("grupo", grupo).groupBy("periodo", "grupo")
        .agg(F.sum("evaluados").alias("evaluados"), prom_ponderado("prom_global").alias("prom_global")))
resto = p7.filter(F.col("grupo") == "RESTO DEL PAIS").select("periodo", F.col("prom_global").alias("resto"))
p7 = p7.join(resto, "periodo").withColumn("diferencia_vs_resto", F.round(F.col("prom_global") - F.col("resto"), 2))
p7.drop("resto").orderBy("periodo", F.desc("prom_global")).show(40, truncate=False)

pdf = p7.toPandas().pivot(index="periodo", columns="grupo", values="prom_global").sort_index()
ax = pdf.plot(kind="bar", figsize=(10, 4.5), rot=0)
ax.set_ylabel("Puntaje global promedio")
ax.set_xlabel("Periodo")
ax.set_title("P7. Ciudades principales frente al resto del país")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.savefig(f"{GRAFICOS}/p7_ciudades.png", dpi=150); plt.close()

+-------+--------------+---------+-----------+-------------------+
|periodo|grupo         |evaluados|prom_global|diferencia_vs_resto|
+-------+--------------+---------+-----------+-------------------+
|20122  |BOGOTA D.C.   |73738    |279.95     |19.81              |
|20122  |MEDELLIN      |25550    |262.89     |2.75               |
|20122  |CALI          |14494    |260.94     |0.8                |
|20122  |RESTO DEL PAIS|393149   |260.14     |0.0                |
|20162  |BOGOTA D.C.   |72528    |277.51     |19.86              |
|20162  |MEDELLIN      |25715    |260.89     |3.24               |
|20162  |CALI          |14255    |259.08     |1.43               |
|20162  |RESTO DEL PAIS|395416   |257.65     |0.0                |
|20172  |BOGOTA D.C.   |73791    |277.16     |19.97              |
|20172  |MEDELLIN      |25595    |260.96     |3.77               |
|20172  |RESTO DEL PAIS|394162   |257.19     |0.0                |
|20172  |CALI          |14682    |256.96     |-0.23           

In [9]:
print("Gráficos generados:", sorted(os.listdir(GRAFICOS)))
spark.stop()

Gráficos generados: ['p1_departamentos.png', 'p2_naturaleza_zona.png', 'p3_areas_region.png', 'p4_evolucion.png', 'p7_ciudades.png']
